In [ ]:
# basedosdados.read_sql
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()

PROJECT_ID = "vertex-sp"
client = bigquery.Client(project=PROJECT_ID)

query = """
SELECT table_name
FROM `basedosdados.br_ms_cnes.INFORMATION_SCHEMA.TABLES`
"""

tabelas = client.query(query).to_dataframe()
print(tabelas)


# ver as colunas das tabelas que interessam
colunas = client.query("""
SELECT table_name, column_name, data_type
FROM `basedosdados.br_ms_cnes.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name IN ('estabelecimento', 'profissional')
ORDER BY table_name, ordinal_position
""").to_dataframe()

print(colunas.to_string())

# CNES-ST (estabelecimentos) de SP, 2 fotografias
query_estabelecimento = """
SELECT
  ano,
  mes,
  sigla_uf,
  id_municipio,
  id_estabelecimento_cnes,
  tipo_unidade,
  id_natureza_juridica,
  quantidade_leito_cirurgico,
  quantidade_leito_clinico,
  quantidade_leito_complementar,
  indicador_atencao_ambulatorial
FROM `basedosdados.br_ms_cnes.estabelecimento`
WHERE sigla_uf = 'SP'
  AND ano IN (2022, 2025)   -- <-- ajuste pros seus 2 anos de snapshot (início/fim dos 4 anos)
  AND mes = 1                -- janeiro como mês de referência da "foto"
"""

df_estabelecimento = client.query(query_estabelecimento).to_dataframe()
print("Linhas:", df_estabelecimento.shape[0])
df_estabelecimento.to_csv("cnes_estabelecimento_sp.csv", index=False)
df_estabelecimento.head()


                      table_name
0                          leito
1                   profissional
2                    equipamento
3                     dicionario
4   estabelecimento_filantropico
5                         equipe
6                estabelecimento
7          servico_especializado
8                   gestao_metas
9                    habilitacao
10              regra_contratual
11        estabelecimento_ensino
12          dados_complementares
13                    incentivos
          table_name                                                 column_name data_type
0    estabelecimento                                                         ano     INT64
1    estabelecimento                                                         mes     INT64
2    estabelecimento                                                    sigla_uf    STRING
3    estabelecimento                                             ano_atualizacao     INT64
4    estabelecimento                              

,ano,mes,sigla_uf,id_municipio,id_estabelecimento_cnes,tipo_unidade,id_natureza_juridica,quantidade_leito_cirurgico,quantidade_leito_clinico,quantidade_leito_complementar,indicador_atencao_ambulatorial
0,2022,1,SP,3550308,3335410,1,1031,0,0,0,1
1,2022,1,SP,3551009,3143309,1,1023,0,0,0,1
2,2022,1,SP,3505401,2093049,1,1244,0,0,0,1
3,2022,1,SP,3514809,2041340,1,1244,0,0,0,1
4,2022,1,SP,3535606,2747960,1,1244,0,0,0,1


In [ ]:
# célula 5 — CNES-PF (profissionais) de SP, mesmas 2 fotografias
query_profissional = """
SELECT
  ano,
  mes,
  sigla_uf,
  id_municipio,
  id_estabelecimento_cnes,
  cartao_nacional_saude,
  cbo_2002,
  tipo_vinculo,
  carga_horaria_ambulatorial
FROM `basedosdados.br_ms_cnes.profissional`
WHERE sigla_uf = 'SP'
  AND ano IN (2022, 2025)   -- <-- mesmos 2 anos da célula anterior
  AND mes = 1
"""

df_profissional = client.query(query_profissional).to_dataframe()
print("Linhas:", df_profissional.shape[0])
df_profissional.to_csv("cnes_profissional_sp.csv", index=False)
df_profissional.head()

Linhas: 2659208


,ano,mes,sigla_uf,id_municipio,id_estabelecimento_cnes,cartao_nacional_saude,cbo_2002,tipo_vinculo,carga_horaria_ambulatorial
0,2022,1,SP,3509502,2081490,980016280314087,131205,010101,0
1,2022,1,SP,3557105,2824868,204311561010009,131210,010101,0
2,2022,1,SP,3552205,2035847,980016281931802,131210,010101,0
3,2022,1,SP,3517406,5119774,207274651840009,131210,010101,40
4,2022,1,SP,3510500,7160984,207273297520009,131210,010101,0


In [ ]:
query_profissional_agg = """
SELECT
  ano,
  sigla_uf,
  id_municipio,
  cbo_2002,
  COUNT(DISTINCT cartao_nacional_saude) AS quantidade_profissionais,
  SUM(carga_horaria_ambulatorial) AS soma_carga_horaria_ambulatorial
FROM `basedosdados.br_ms_cnes.profissional`
WHERE sigla_uf = 'SP'
  AND ano BETWEEN 2021 AND 2024
  AND mes = 1
GROUP BY ano, sigla_uf, id_municipio, cbo_2002
"""

df_profissional_agg = client.query(query_profissional_agg).to_dataframe()
print("Linhas:", df_profissional_agg.shape[0])
df_profissional_agg.to_csv("cnes_profissional_sp_agg.csv", index=False)
df_profissional_agg.head()

Linhas: 179903


,ano,sigla_uf,id_municipio,cbo_2002,quantidade_profissionais,soma_carga_horaria_ambulatorial
0,2022,SP,3543907,142105,48,0
1,2022,SP,3509502,221205,222,5018
2,2022,SP,3550308,223405,3588,77066
3,2022,SP,3509601,223505,53,1489
4,2022,SP,3549706,223505,64,1390


In [ ]:
query_profissional_agg = """
SELECT
  ano,
  sigla_uf,
  id_municipio,
  SUBSTR(cbo_2002, 1, 4) AS familia_ocupacional_cbo,
  COUNT(DISTINCT cartao_nacional_saude) AS quantidade_profissionais,
  SUM(carga_horaria_ambulatorial) AS soma_carga_horaria_ambulatorial
FROM `basedosdados.br_ms_cnes.profissional`
WHERE sigla_uf = 'SP'
  AND ano BETWEEN 2021 AND 2024
  AND mes = 1
GROUP BY ano, sigla_uf, id_municipio, familia_ocupacional_cbo
"""

df_profissional_agg = client.query(query_profissional_agg).to_dataframe()
print("Linhas:", df_profissional_agg.shape[0])
df_profissional_agg.to_csv("cnes_profissional_sp_agg.csv", index=False)
df_profissional_agg.head()

Linhas: 94088


,ano,sigla_uf,id_municipio,familia_ocupacional_cbo,quantidade_profissionais,soma_carga_horaria_ambulatorial
0,2021,SP,3550308,3221,54,535
1,2021,SP,3509205,1114,1,0
2,2021,SP,3550308,2030,220,345
3,2021,SP,3505708,2523,3,0
4,2022,SP,3505609,7832,1,0


In [ ]:


# célula única — extrai, exporta e baixa CNES-ST (estabelecimento) e CNES-PF (profissional agregado) de uma vez

# 1) Estabelecimentos (CNES-ST)
query_estabelecimento = """
SELECT
  ano,
  mes,
  sigla_uf,
  id_municipio,
  id_estabelecimento_cnes,
  tipo_unidade,
  id_natureza_juridica,
  quantidade_leito_cirurgico,
  quantidade_leito_clinico,
  quantidade_leito_complementar,
  indicador_atencao_ambulatorial
FROM `basedosdados.br_ms_cnes.estabelecimento`
WHERE sigla_uf = 'SP'
  AND ano BETWEEN 2021 AND 2024
  AND mes = 1
"""

df_estabelecimento = client.query(query_estabelecimento).to_dataframe()
df_estabelecimento.to_csv("cnes_estabelecimento_sp.csv", index=False)
print("Estabelecimento — linhas:", df_estabelecimento.shape[0])

# 2) Profissionais (CNES-PF), agregado por família ocupacional do CBO
query_profissional_agg = """
SELECT
  ano,
  sigla_uf,
  id_municipio,
  SUBSTR(cbo_2002, 1, 4) AS familia_ocupacional_cbo,
  COUNT(DISTINCT cartao_nacional_saude) AS quantidade_profissionais,
  SUM(carga_horaria_ambulatorial) AS soma_carga_horaria_ambulatorial
FROM `basedosdados.br_ms_cnes.profissional`
WHERE sigla_uf = 'SP'
  AND ano BETWEEN 2021 AND 2024
  AND mes = 1
GROUP BY ano, sigla_uf, id_municipio, familia_ocupacional_cbo
"""

df_profissional_agg = client.query(query_profissional_agg).to_dataframe()
df_profissional_agg.to_csv("cnes_profissional_sp_agg.csv", index=False)
print("Profissional (agregado) — linhas:", df_profissional_agg.shape[0])

# 3) Baixar os dois arquivos pro seu computador
from google.colab import files
files.download("cnes_estabelecimento_sp.csv")
files.download("cnes_profissional_sp_agg.csv")

Estabelecimento — linhas: 331052
Profissional (agregado) — linhas: 94088


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# célula 7 — descobrir os datasets e tabelas do SIASUS e do IBGE população
datasets = list(client.list_datasets(project="basedosdados"))

palavras_chave = ["sia", "populac", "ibge"]

for d in datasets:
    nome = d.dataset_id.lower()
    if any(p in nome for p in palavras_chave):
        print(f"\n=== dataset: {d.dataset_id} ===")
        tabelas = client.list_tables(f"basedosdados.{d.dataset_id}")
        for t in tabelas:
            print(" -", t.table_id)



            # célula 8 — colunas do SIASUS e da população do IBGE, de uma vez
colunas_siasus = client.query("""
SELECT table_name, column_name, data_type
FROM `basedosdados.br_ms_sia.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'producao_ambulatorial'
ORDER BY ordinal_position
""").to_dataframe()

colunas_populacao = client.query("""
SELECT table_name, column_name, data_type
FROM `basedosdados.br_ibge_populacao.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'municipio'
ORDER BY ordinal_position
""").to_dataframe()

print("=== SIASUS (producao_ambulatorial) ===")
print(colunas_siasus.to_string())
print("\n=== IBGE (populacao.municipio) ===")
print(colunas_populacao.to_string())


=== dataset: br_ibge_amc ===
 - municipio_de_para

=== dataset: br_ibge_cbo_2002 ===
 - perfil_ocupacional
 - sinonimo

=== dataset: br_ibge_censo_2022 ===
 - alfabetizacao_grupo_idade_sexo_raca
 - cadastro_enderecos
 - caracteristica_domicilio_grupo_idade_raca_destino_lixo
 - caracteristica_domicilio_grupo_idade_raca_esgotamento_sanitario
 - caracteristica_domicilio_grupo_idade_raca_ligacao_abastecimento_agua
 - caracteristica_domicilio_grupo_idade_raca_tipo_domicilio
 - dicionario
 - domicilio_recenseado
 - indice_envelhecimento_raca
 - municipio
 - populacao_grupo_idade_sexo_raca
 - populacao_grupo_idade_uf
 - populacao_idade_sexo
 - setor_censitario
 - terra_indigena
 - territorio_quilombola

=== dataset: br_ibge_censo_demografico ===
 - dicionario
 - microdados_domicilio_1970
 - microdados_domicilio_1980
 - microdados_domicilio_1991
 - microdados_domicilio_2000
 - microdados_domicilio_2010
 - microdados_pessoa_1970
 - microdados_pessoa_1980
 - microdados_pessoa_1991
 - microdados

In [ ]:
# célula 9 — extrair, exportar e baixar SIASUS e IBGE de uma vez

# 1) SIASUS — atendimentos, agregados por município + ano + mês (mantém a variação mensal real)
query_siasus = """
SELECT
  ano,
  mes,
  sigla_uf,
  id_municipio,
  SUM(quantidade_aprovada_procedimento) AS quantidade_atendimentos,
  SUM(valor_aprovado_procedimento) AS valor_total_aprovado
FROM `basedosdados.br_ms_sia.producao_ambulatorial`
WHERE sigla_uf = 'SP'
  AND ano BETWEEN 2021 AND 2024
GROUP BY ano, mes, sigla_uf, id_municipio
"""

df_siasus = client.query(query_siasus).to_dataframe()
df_siasus.to_csv("siasus_atendimentos_sp.csv", index=False)
print("SIASUS — linhas:", df_siasus.shape[0])

# 2) IBGE — população por município e ano
query_populacao = """
SELECT
  ano,
  sigla_uf,
  id_municipio,
  populacao
FROM `basedosdados.br_ibge_populacao.municipio`
WHERE sigla_uf = 'SP'
  AND ano BETWEEN 2021 AND 2024
"""

df_populacao = client.query(query_populacao).to_dataframe()
df_populacao.to_csv("ibge_populacao_sp.csv", index=False)
print("IBGE população — linhas:", df_populacao.shape[0])

# 3) baixar os dois
from google.colab import files
files.download("siasus_atendimentos_sp.csv")
files.download("ibge_populacao_sp.csv")


SIASUS — linhas: 28909
IBGE população — linhas: 2580


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>